# 生成策略速查表(精简版)

> 本文是 [ch07.ipynb](./ch07.ipynb) 的浓缩版。只保留核心公式和参数表,方便快速查阅。

## Greedy → Sampling 一行代码

```python
# Greedy(确定性)
next_token = torch.argmax(logits, dim=-1)

# Sampling(随机性)
probs = torch.softmax(logits / temperature, dim=-1)
next_token = torch.multinomial(probs, num_samples=1)
```

当 `temperature → 0` 时,softmax 变成 one-hot,sampling 退化为 greedy。

## KV Cache 原理图

```
无 cache(朴素):               有 cache:
                                
Step 0: forward([A,B,C])       Step 0: forward([A,B,C]) → 缓存 K₀₁₂ V₀₁₂
Step 1: forward([A,B,C,D])     Step 1: forward([D]) + cache → 拼接 K₃ V₃
Step 2: forward([A,B,C,D,E])   Step 2: forward([E]) + cache → 拼接 K₄ V₄
                                
O(T²)                          O(T)

每步重新算所有 token            每步只算 1 个新 token
```

## 参数速查表

| 参数 | 默认值 | 作用 | 推荐范围 |
|---|---|---|---|
| `temperature` | 0.85 | 控制分布锐度 | 0.7-0.9(聊天)/ 0.1-0.3(事实) |
| `top_p` | 0.85 | 累积概率截断 | 0.8-0.95 |
| `top_k` | 50 | 固定数量截断 | vocab 大时用;小词表设 0 禁用 |
| `repetition_penalty` | 1.0 | 惩罚重复 token | 1.0-1.3(>1.5 过度) |
| `do_sample` | True | True=采样 / False=greedy | 聊天 True / 事实 False |
| `max_new_tokens` | 8192 | 最大生成长度 | 按需设 |
| `use_cache` | True | 是否启用 KV cache | **永远 True** |

> **执行顺序**:`/temperature` → repetition penalty → top_k filter → top_p filter → `multinomial` 采样

In [ ]:
# generate 参数速查 + 典型配置预设
PRESETS = {
    "精确问答(低随机)": dict(temperature=0.1, top_p=0.9, do_sample=True),
    "日常聊天(平衡)":   dict(temperature=0.85, top_p=0.85, do_sample=True),
    "创意写作(高多样)": dict(temperature=1.0, top_p=0.95, do_sample=True),
    "Greedy(确定)":     dict(temperature=1.0, do_sample=False),
}

print("=== 典型生成配置预设 ===\n")
for name, params in PRESETS.items():
    print(f"{name}:")
    print(f"  {params}\n")

print("调用方式:")
print("  model.generate(input_ids, **PRESETS['日常聊天(平衡)'])")